In [0]:
%run ./utils

In [0]:
from datetime import datetime, timedelta
import pyspark.sql.functions as F

def get_checksum_df(clear_consumer_table, master_consumer_table, start_time, end_time):
    clear_df = (spark.table(clear_consumer_table)
        .filter((F.col("SRCC_UPDATE_DT") >= F.lit(start_time)) & (F.col("SRCC_UPDATE_DT") <= F.lit(end_time)))
        .filter(F.col("IS_INCLUDE") == True) #仅check regular 数据
        .select(
            F.col("SRCC_ID"),
            F.col("SRCC_MRKT_CODE").alias("Market"),
            F.col("SRCC_BRND_CODE").alias("Brand"),
            F.col("SRCC_SRCS_CODE").alias("SourceSystemCode"),
            F.col("SRCC_CONSUMERID").alias("ConsumerId"),
            F.col("SRCC_CREATION_DT").alias("creationTime"),
            F.col("TASK_ID").alias("TASK_ID")
        )
        .distinct())

    master_df = spark.table(master_consumer_table)

    check_df = (clear_df.alias("clear_df")
        .join(master_df.alias("master_df"), 
            (F.col("clear_df.Market") == F.col("master_df.scon_mrkt_code")) &
            (F.col("clear_df.Brand") == F.col("master_df.scon_brnd_code")) &
            (F.col("clear_df.SourceSystemCode") == F.col("master_df.scon_srcs_code")) &
            (F.col("clear_df.ConsumerId") == F.col("master_df.scon_consumerid")),
            "left"
        )
        .filter(F.col("master_df.scon_mrkt_code").isNull())
        .select(F.col("clear_df.*"))
        .distinct()
    )
    
    return check_df


In [0]:
def monitor_main(monitor_id, clear_consumer_table, master_consumer_table, start_time, end_time, max_rows=MAX_ROWS, max_cols=MAX_COLS, to_addrs=None):
    # 1. check sum
    check_df = get_checksum_df(clear_consumer_table, master_consumer_table, start_time, end_time)
    check_df.cache()

    if check_df.count() > 0:
        print(f"This inspection found invalid data: {monitor_id}")
        display(check_df)

        # 2. build email body
        html_body = build_html_table_from_spark_df(check_df, max_rows=max_rows, max_cols=max_cols)

        recipients = to_addrs or TO_ADDRS
        if not recipients:
            raise ValueError("to_addrs is empty; no recipients configured for the consumer ID missing report email.")

        send_email(
            subject=SUBJECT.format(yyyymmdd=end_time.strftime("%Y%m%d")),
            html_body=html_body,
            to_addrs=recipients,
            cc_addrs=CC_ADDRS,
            bcc_addrs=BCC_ADDRS,
            custom_text = f"This inspection found miss cid.  <br>Check time period(UTC): {start_time} -> {end_time}. <br>Check the table: {clear_consumer_table} -> {master_consumer_table}. <br>monitor_id: {monitor_id}"
        )

    else:
        print(f"There is no invalid data in this check: {monitor_id}")

    check_df.unpersist()

In [0]:
TO_ADDRS: List[str] = []
CC_ADDRS: List[str] = []
BCC_ADDRS: List[str] = []

SUBJECT = "[Major] [MDM] Integrity Check - Record Missing in Master {yyyymmdd}"

In [0]:
monitor_id = dbutils.widgets.get("monitor_id")
hour_time_period = int(dbutils.widgets.get("hour_time_period"))
clear_consumer_table = dbutils.widgets.get("clear_consumer_table")
master_consumer_table = dbutils.widgets.get("master_consumer_table")

try:
    trigger_timestamp_ms = int(dbutils.widgets.get("trigger_timestamp_ms")) / 1000
except:
    trigger_timestamp_ms = int(datetime.now().timestamp())

# Maximum rows/columns to show in the email HTML tables.
try:
    max_rows = int(dbutils.widgets.get("max_rows"))
except:
    max_rows = MAX_ROWS

try:
    max_cols = int(dbutils.widgets.get("max_cols"))
except:
    max_cols = MAX_COLS

# Comma-separated list of recipient email addresses.
to_addrs_str = dbutils.widgets.get("to_addrs")
to_addrs = [x.strip() for x in to_addrs_str.split(",") if x.strip()] if to_addrs_str else TO_ADDRS

end_time = datetime.fromtimestamp(trigger_timestamp_ms)
start_time = end_time - timedelta(hours= hour_time_period)

print(f"monitor_id: {monitor_id}")
print(f"clear_consumer_table: {clear_consumer_table}")
print(f"master_consumer_table: {master_consumer_table}")
print(f"hour_time_period: {hour_time_period}")
print(f"max_rows: {max_rows}, max_cols: {max_cols}")
print(f"to_addrs: {to_addrs}")
print(f"start_time: {start_time}, end_time: {end_time}")

monitor_main(monitor_id, clear_consumer_table, master_consumer_table, start_time, end_time, max_rows, max_cols, to_addrs)

